# Realtime Data & WebSockets

The photo app now has a REST API and a working Flet frontend. Triggering an S3 index job and waiting for it to finish, however, means staring at a spinner with no feedback — no idea whether 10 photos have been processed or 10,000. The fundamental problem is that REST is request/response: the client asks, the server answers, and then the connection closes. Any change that happens on the server between requests is invisible to the client unless it asks again. This notebook adds the missing layer: **realtime push delivery** so that the backend can stream live progress events to the UI as they happen.

We cover the three dominant push patterns, build a pub/sub fan-out manager in FastAPI, wire it to the S3 indexing pipeline from notebook 08, and add a Server-Sent Events endpoint as a lightweight alternative. The Flet progress UI that consumes these streams is presented as a static code block.

## Event-Driven Architecture

**Request/response vs. push.** In the request/response model the client initiates every interaction: it sends a request, the server processes it, and the connection closes. This works well for user-triggered queries. It breaks down for server-initiated updates, for instance a background indexing job that makes progress without any user prompt. The naive fix is **polling**: the client repeatedly asks "are you done yet?" on a fixed interval.

<br>

**The polling anti-pattern.** Polling wastes resources on both sides. A 1-second poll interval for a 10-minute job generates 600 HTTP requests, most of which receive the same "still running" response. Each request pays the full cost of TCP connection (or HTTP keep-alive overhead), TLS handshake if applicable, request parsing, handler dispatch, and response serialization — regardless of whether any new information exists. At scale, a fleet of polling clients can saturate the server with traffic that carries no signal.

<br>

**Three push patterns.** We compare the dominant alternatives:

| Pattern | Direction | Transport | Best for |
|---|---|---|---|
| **WebSocket** | Bidirectional | TCP (after HTTP upgrade) | Chat, collaboration, live cursors |
| **Server-Sent Events** | Server → Client | HTTP/1.1 long response | Progress bars, log streams, dashboards |
| **Long polling** | Server → Client | HTTP | Simple cases, firewalled environments |

WebSocket is the most capable: after a one-time HTTP upgrade handshake, both client and server can send messages at any time over a single persistent TCP connection. SSE is simpler: it is just a long HTTP response the server keeps open and drips data into; the browser's `EventSource` API reconnects automatically if the connection drops. Long polling holds the HTTP request open until the server has something to say, then closes it; the client immediately re-opens. We use WebSocket as the primary mechanism and SSE as the secondary.

:::{.callout-note}
SSE works over HTTP/2 multiplexing without any special protocol support. WebSocket over HTTP/2 requires RFC 8441 (`CONNECT` upgrade), which is not universally supported. For pure server-push workloads, SSE is often the pragmatic choice.

:::

Demonstrating the polling anti-pattern: counting requests per minute for a job that takes 60 seconds:

In [ ]:
import asyncio
import time

async def poll_status(interval_s: float, job_duration_s: float) -> dict:
    """Simulate polling a /status endpoint until the job completes."""
    start = time.perf_counter()
    request_count = 0

    while True:
        elapsed = time.perf_counter() - start
        request_count += 1
        done = elapsed >= job_duration_s

        if done:
            break

        await asyncio.sleep(interval_s)

    total_s = time.perf_counter() - start
    return {
        "requests": request_count,
        "requests_per_minute": round(request_count / total_s * 60, 1),
        "wasted_requests": request_count - 1,  # all but the final one carried no new info
    }

# 1-second polling interval, 10-second simulated job
stats = asyncio.run(poll_status(interval_s=1.0, job_duration_s=10.0))
print(stats)

## WebSocket Protocol

**The upgrade handshake.** A WebSocket connection begins as an ordinary HTTP/1.1 `GET` request with two special headers: `Upgrade: websocket` and `Connection: Upgrade`. The server responds with `101 Switching Protocols`, and from that moment the TCP socket is no longer HTTP — it speaks the WebSocket framing protocol instead. No new connection is opened; the existing socket is repurposed.

<br>

**Framing.** WebSocket transmits data as a sequence of **frames**. Each frame has a 2-byte minimum header encoding the **opcode** and payload length. Key opcodes:

| Opcode | Meaning |
|--------|--------|
| `0x1` | Text frame: UTF-8 payload |
| `0x2` | Binary frame: raw bytes |
| `0x8` | Close: initiates the closing handshake |
| `0x9` | Ping: keep-alive probe from either side |
| `0xA` | Pong: response to a ping |

We send JSON as text frames throughout this notebook.

<br>

**FastAPI WebSocket API.** FastAPI exposes WebSocket connections through a `WebSocket` object injected into route handlers. The connection lifecycle maps directly onto four awaitable calls:

```python
await ws.accept()        # complete the upgrade handshake
await ws.send_text(data) # send a UTF-8 text frame
await ws.receive_text()  # receive a text frame (blocks until one arrives)
await ws.close()         # send a Close frame and tear down
```

The route is decorated with `@app.websocket("/ws/{channel}")` instead of the usual `@app.get`.

<br>

**Connection lifecycle.** Client connects → server calls `accept()` → messages flow in both directions → either side sends a Close frame → the other side acknowledges → the TCP connection is torn down. Any unhandled exception in the handler also closes the connection; the client will see a `1011 Internal Error` close code.

:::{.callout-caution}
Always `await ws.accept()` before any send or receive. Sending to an unaccepted WebSocket raises a `RuntimeError` in Starlette that is easy to miss in tests where the transport is mocked.

:::

Minimal echo server and a `websockets` client that sends three messages:

In [ ]:
from fastapi import FastAPI, WebSocket
from fastapi.testclient import TestClient

echo_app = FastAPI()

@echo_app.websocket("/ws/echo")
async def echo_handler(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            data = await ws.receive_text()
            await ws.send_text(f"echo: {data}")
    except Exception:
        pass

Testing the echo server with Starlette's `TestClient`:

In [ ]:
client = TestClient(echo_app)

with client.websocket_connect("/ws/echo") as ws:
    for i in range(3):
        ws.send_text(f"message {i}")
        reply = ws.receive_text()
        print(reply)

## Connection Manager (Pub/Sub Fan-out)

A single WebSocket connection is sufficient for one client watching one job. We need a **pub/sub fan-out layer** when multiple clients may subscribe to the same channel (two browser tabs open on the same job) or when a single background task must broadcast to an arbitrary number of subscribers without knowing who they are.

<br>

**`ConnectionManager` design.** We maintain a `dict[str, set[WebSocket]]` mapping a channel ID (here: `job_id`) to the set of currently connected clients. Three operations:

- `connect(channel, ws)`: adds `ws` to the channel's set after accepting the connection.
- `disconnect(channel, ws)`: removes `ws`; if the set becomes empty, deletes the channel key.
- `broadcast(channel, message)`: sends `message` to every connected client; removes any that raise `WebSocketDisconnect`.

**Thread safety.** FastAPI runs on an `asyncio` event loop. Concurrent coroutines are multiplexed cooperatively (no OS threads by default), so a plain `dict` is safe as long as we never `await` in the middle of a mutation. Where we must `await` mid-operation (e.g., inside the broadcast loop), we protect the outer dict with an `asyncio.Lock`.

Full `ConnectionManager` implementation:

In [ ]:
import asyncio
from fastapi import WebSocket
from fastapi.websockets import WebSocketDisconnect


class ConnectionManager:
    """Pub/sub fan-out over WebSocket connections, keyed by channel ID."""

    def __init__(self):
        self._channels: dict[str, set[WebSocket]] = {}
        self._lock = asyncio.Lock()

    async def connect(self, channel: str, ws: WebSocket) -> None:
        await ws.accept()
        async with self._lock:
            self._channels.setdefault(channel, set()).add(ws)

    async def disconnect(self, channel: str, ws: WebSocket) -> None:
        async with self._lock:
            subs = self._channels.get(channel, set())
            subs.discard(ws)
            if not subs:
                self._channels.pop(channel, None)

    async def broadcast(self, channel: str, message: str) -> None:
        dead: list[WebSocket] = []
        async with self._lock:
            subs = list(self._channels.get(channel, set()))

        for ws in subs:
            try:
                await ws.send_text(message)
            except (WebSocketDisconnect, Exception):
                dead.append(ws)

        for ws in dead:
            await self.disconnect(channel, ws)

    def subscriber_count(self, channel: str) -> int:
        return len(self._channels.get(channel, set()))


manager = ConnectionManager()
print("ConnectionManager ready")

Unit test: three mock WebSocket objects each subscribe to `"job-42"`, a broadcast is sent, and we verify all three received the message:

In [ ]:
from unittest.mock import AsyncMock, MagicMock


async def test_broadcast():
    mgr = ConnectionManager()
    channel = "job-42"

    # Create three mock WebSocket objects
    mocks = [MagicMock(spec=WebSocket) for _ in range(3)]
    for m in mocks:
        m.accept = AsyncMock()
        m.send_text = AsyncMock()
        await mgr.connect(channel, m)

    assert mgr.subscriber_count(channel) == 3

    await mgr.broadcast(channel, '{"processed": 100}')

    for m in mocks:
        m.send_text.assert_awaited_once_with('{"processed": 100}')

    print("All 3 subscribers received the broadcast ✓")


asyncio.run(test_broadcast())

## Live Indexing Progress

The S3 pipeline from notebook 08 processes photos in batches. After each batch it emits a `ProgressEvent` describing the current state of the job. We model this as a Pydantic schema so it serialises cleanly to JSON for transmission over the WebSocket:

```python
{"job_id": "abc123", "total": 1000, "processed": 450, "failed": 2, "status": "running"}
```

**Flow.** `POST /pipeline/start` kicks off the indexing task as a FastAPI `BackgroundTasks` job and immediately returns a `job_id` to the caller. The background task runs independently: it scans S3, downloads photos, computes CLIP embeddings, and upserts rows. After each batch it calls `manager.broadcast(job_id, event.model_dump_json())`. Any client that has opened the `GET /pipeline/{job_id}/ws` WebSocket endpoint will receive each event as it is emitted — no polling required.

Defining the `ProgressEvent` schema and the pipeline router:

In [ ]:
import uuid
import asyncio
from enum import Enum
from pydantic import BaseModel
from fastapi import APIRouter, BackgroundTasks, WebSocket
from fastapi.websockets import WebSocketDisconnect


class JobStatus(str, Enum):
    pending  = "pending"
    running  = "running"
    done     = "done"
    failed   = "failed"


class ProgressEvent(BaseModel):
    job_id:    str
    total:     int
    processed: int
    failed:    int
    status:    JobStatus


pipeline_router = APIRouter(prefix="/pipeline", tags=["pipeline"])


@pipeline_router.post("/start")
async def start_pipeline(background_tasks: BackgroundTasks) -> dict:
    job_id = str(uuid.uuid4())[:8]
    background_tasks.add_task(run_pipeline, job_id)
    return {"job_id": job_id}


@pipeline_router.websocket("/{job_id}/ws")
async def pipeline_ws(job_id: str, ws: WebSocket):
    await manager.connect(job_id, ws)
    try:
        while True:
            await ws.receive_text()   # keep the connection open; client sends nothing
    except WebSocketDisconnect:
        await manager.disconnect(job_id, ws)


print("Pipeline router defined")

Mock pipeline background task emitting 10 progress events:

In [ ]:
async def run_pipeline(job_id: str, total: int = 1000, batches: int = 10) -> None:
    """Mock S3 indexing pipeline; broadcasts a ProgressEvent after each batch."""
    batch_size = total // batches
    failed = 0

    for i in range(1, batches + 1):
        await asyncio.sleep(0.1)           # simulate I/O work
        processed = i * batch_size
        if i == 3:                         # inject one failed item for realism
            failed += 1

        status = JobStatus.running if i < batches else JobStatus.done
        event = ProgressEvent(
            job_id=job_id,
            total=total,
            processed=processed,
            failed=failed,
            status=status,
        )
        await manager.broadcast(job_id, event.model_dump_json())
        print(f"[{job_id}] batch {i}/{batches}: {event.model_dump_json()}")


asyncio.run(run_pipeline("demo-job"))

Simulating a WebSocket client that subscribes and prints each event as it arrives:

In [ ]:
import json


async def mock_ws_client(job_id: str, max_events: int = 10):
    """Simulate a WebSocket client collecting progress events via a shared queue."""
    queue: asyncio.Queue[str] = asyncio.Queue()

    # Monkey-patch a lightweight subscriber onto the manager
    from unittest.mock import AsyncMock, MagicMock
    ws = MagicMock(spec=WebSocket)
    ws.accept = AsyncMock()
    ws.send_text = AsyncMock(side_effect=lambda msg: queue.put_nowait(msg))

    await manager.connect(job_id, ws)
    await run_pipeline(job_id)

    events = []
    while not queue.empty():
        raw = await queue.get()
        events.append(ProgressEvent.model_validate_json(raw))

    return events


received = asyncio.run(mock_ws_client("test-job"))
print(f"Received {len(received)} events; final status: {received[-1].status}")

## Server-Sent Events

**SSE over HTTP/1.1.** Server-Sent Events use a long-lived HTTP response with `Content-Type: text/event-stream`. The server writes newline-delimited records in the format `data: {payload}\n\n` and keeps the response body open. The browser's built-in `EventSource` API receives each `data:` line as a JavaScript event. Reconnection is handled automatically by the browser — if the TCP connection drops, `EventSource` re-issues the `GET` request, optionally including a `Last-Event-ID` header so the server can resume from where it left off.

<br>

**FastAPI `StreamingResponse`.** In FastAPI, SSE is implemented with a `StreamingResponse` wrapping an `async` generator that `yield`s each event string. The response stays open as long as the generator has not returned. The `media_type` must be set to `"text/event-stream"` and the `Cache-Control: no-cache` header prevents intermediate proxies from buffering the stream:

```python
async def event_stream(job_id: str):
    async for event in get_events(job_id):
        yield f"data: {event}\n\n"

return StreamingResponse(event_stream(job_id), media_type="text/event-stream")
```

<br>

**When SSE beats WebSocket.** SSE requires no protocol upgrade, no special CORS handling beyond standard HTTP, and works over HTTP/2 multiplexing. For pure server-push use cases — progress bars, log tailing, real-time dashboards — SSE is simpler to deploy and debug. The limitation is unidirectionality: the client cannot send messages back over the same connection without opening a separate HTTP request.

:::{.callout-note}
SSE has a browser-side limit of 6 concurrent connections per origin under HTTP/1.1 (the same limit as any HTTP connection). Under HTTP/2 this limit does not apply because connections are multiplexed. For server-to-client log streaming in a desktop Flet app, neither limit is relevant — the Flet app uses the Python `httpx` async client, not a browser.

:::

Defining the SSE endpoint with `StreamingResponse`:

In [ ]:
import asyncio
import json
from fastapi import APIRouter
from fastapi.responses import StreamingResponse


sse_router = APIRouter(prefix="/pipeline", tags=["pipeline-sse"])


async def _sse_event_stream(job_id: str):
    """Async generator that yields SSE-formatted progress events."""
    total, batches = 500, 5
    batch_size = total // batches

    for i in range(1, batches + 1):
        await asyncio.sleep(0.05)
        event = ProgressEvent(
            job_id=job_id,
            total=total,
            processed=i * batch_size,
            failed=0,
            status=JobStatus.running if i < batches else JobStatus.done,
        )
        yield f"data: {event.model_dump_json()}\n\n"


@sse_router.get("/{job_id}/events")
async def pipeline_sse(job_id: str):
    return StreamingResponse(
        _sse_event_stream(job_id),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )


print("SSE endpoint defined")

Consuming the SSE stream with an `httpx` streaming client:

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

sse_app = FastAPI()
sse_app.include_router(sse_router)

client = TestClient(sse_app)

events = []
with client.stream("GET", "/pipeline/job-sse-demo/events") as resp:
    for line in resp.iter_lines():
        if line.startswith("data:"):
            payload = line.removeprefix("data: ")
            event = ProgressEvent.model_validate_json(payload)
            events.append(event)
            print(f"SSE: processed={event.processed}/{event.total} status={event.status}")

print(f"\nTotal SSE events received: {len(events)}")

## Flet Progress UI

**Task.** Build an `IndexingProgress` Flet component that subscribes to the WebSocket endpoint and renders a `ProgressBar` plus a stats row. The component updates live as events arrive and shows a snackbar on completion.

Since Flet UI rendering cannot execute inside a Jupyter kernel, we present the component as a static file block:

```{.python filename="src/components/indexing_progress.py"}
import json
import threading
import websocket  # websocket-client library
import flet as ft


class IndexingProgress(ft.Column):
    """Live indexing progress component backed by a WebSocket subscription."""

    def __init__(self, job_id: str, api_base: str = "ws://localhost:8000"):
        super().__init__()
        self.job_id   = job_id
        self.api_base = api_base

        self.bar      = ft.ProgressBar(width=400, value=0)
        self.label    = ft.Text("Waiting for progress...")
        self.chip     = ft.Chip(label=ft.Text("pending"), disabled=True)
        self.controls = [self.bar, self.label, self.chip]

    def did_mount(self):
        threading.Thread(target=self._listen, daemon=True).start()

    def _listen(self):
        url = f"{self.api_base}/pipeline/{self.job_id}/ws"
        ws = websocket.WebSocketApp(
            url,
            on_message=self._on_message,
            on_error=lambda ws, err: print(f"WS error: {err}"),
        )
        ws.run_forever()

    def _on_message(self, ws_app, raw: str):
        event = json.loads(raw)
        total     = event["total"]
        processed = event["processed"]
        failed    = event["failed"]
        status    = event["status"]

        def update(_):
            self.bar.value    = processed / total if total else 0
            self.label.value  = (
                f"{processed:,} / {total:,} processed  •  {failed} failed"
            )
            self.chip.label.value = status

            if status == "done":
                self.page.show_snack_bar(
                    ft.SnackBar(
                        content=ft.Text(
                            f"Indexing complete — {processed - failed:,} photos added"
                        ),
                        open=True,
                    )
                )
            self.update()

        self.page.run_task(update, None)
```

The component subscribes on `did_mount` (called when the control is first added to the page) in a daemon thread. The `_on_message` callback schedules `update` back on the Flet event loop via `page.run_task`, keeping UI mutations on the correct thread.

## Appendix: Backpressure & Slow Consumers

**The problem.** A fast producer — say, the CLIP embedding pipeline processing 50 photos per second — emits WebSocket messages far faster than a slow consumer (a mobile client on a congested network) can acknowledge them. Without any throttling, the pending messages accumulate in memory on the server side. For a long-running job, this can exhaust memory or cause significant latency spikes.

<br>

**Bounded queue as a buffer.** We place an `asyncio.Queue(maxsize=N)` between the producer and the WebSocket sender. The producer calls `queue.put_nowait(event)`. If the queue is full — because the consumer has not drained it fast enough — `put_nowait` raises `asyncio.QueueFull`. At that point we have a choice: drop the oldest item (ring-buffer semantics, suitable for progress bars where only the latest value matters), raise back-pressure by making the producer `await queue.put(event)` (suspends the producer), or drop the newest item (appropriate when older state is more important to preserve).

For a progress bar, dropping the oldest item is correct: if the consumer is slow we want it to see the *current* state when it catches up, not a stale intermediate state.

Demonstrating queue backpressure: producer at 20 events/s, consumer at 5 events/s:

In [ ]:
import asyncio


async def backpressure_demo(
    producer_rate: float = 20,   # events per second
    consumer_rate: float = 5,    # events per second
    n_events: int = 20,
    queue_size: int = 4,
):
    queue: asyncio.Queue[int] = asyncio.Queue(maxsize=queue_size)
    dropped = 0
    consumed = []

    async def producer():
        nonlocal dropped
        for i in range(n_events):
            try:
                queue.put_nowait(i)            # <1>
            except asyncio.QueueFull:
                queue.get_nowait()             # <2>
                queue.put_nowait(i)
                dropped += 1
            await asyncio.sleep(1 / producer_rate)

    async def consumer():
        while True:
            try:
                item = queue.get_nowait()
                consumed.append(item)
            except asyncio.QueueEmpty:
                pass
            await asyncio.sleep(1 / consumer_rate)
            if len(consumed) >= n_events - dropped and queue.empty():
                break

    await asyncio.gather(producer(), consumer())
    print(f"Produced: {n_events}  Consumed: {len(consumed)}  Dropped (oldest): {dropped}")
    print(f"Consumer always sees latest state: last consumed = {consumed[-1]}")


asyncio.run(backpressure_demo())

1. Try to enqueue without blocking.
2. Queue is full: evict the oldest pending event (drop it) and insert the new one. This ensures the consumer always sees fresh state when it eventually drains the queue.

---

■